# 第一章：基础提示结构

- [课程](#lesson)
- [练习](#exercises)
- [示例演练场](#example-playground)

## 设置

运行以下设置单元格以加载您的 API key 并建立 `get_completion` 辅助函数。

In [ ]:
!pip install anthropic

# Import python's built-in regular expression library
import re
import anthropic

# Retrieve the API_KEY & MODEL_NAME variables from the IPython store
%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def get_completion(prompt: str, system_prompt=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        system=system_prompt,
        messages=[
          {"role": "user", "content": prompt}
        ]
    )
    return message.content[0].text

---

## 课程

Anthropic 提供两个 API，旧版的 [Text Completions API](https://docs.anthropic.com/claude/reference/complete_post) 和当前的 [Messages API](https://docs.anthropic.com/claude/reference/messages_post)。在本教程中，我们将专门使用 Messages API。

使用 Messages API 调用 Claude 至少需要以下参数：
- `model`：您打算调用的模型的 [API 模型名称](https://docs.anthropic.com/claude/docs/models-overview#model-recommendations)

- `max_tokens`：停止生成之前要生成的最大 token 数。请注意，Claude 可能会在达到此最大值之前停止。此参数仅指定要生成的绝对最大 token 数。此外，这是一个*硬*停止，意味着它可能导致 Claude 在单词或句子中间停止生成。

- `messages`：输入消息数组。我们的模型经过训练，可以在 `user` 和 `assistant` 交替对话轮次中运行。创建新的 `Message` 时，您使用 messages 参数指定之前的对话轮次，然后模型会在对话中生成下一个 `Message`。
  - 每个输入消息都必须是一个包含 `role` 和 `content` 的对象。您可以指定单个 `user` 角色消息，或者可以包含多个 `user` 和 `assistant` 消息（如果是这样，它们必须交替）。第一条消息必须始终使用 user `role`。

还有一些可选参数，例如：
- `system`：系统提示 - 下面会详细介绍。
  
- `temperature`：Claude 响应的可变性程度。在这些课程和练习中，我们将 `temperature` 设置为 0。

有关所有 API 参数的完整列表，请访问我们的 [API 文档](https://docs.anthropic.com/claude/reference/messages_post)。

### 示例

让我们看看 Claude 如何响应一些格式正确的提示。对于以下每个单元格，运行该单元格（`shift+enter`），Claude 的响应将出现在代码块下方。

In [ ]:
# Prompt
PROMPT = "Hi Claude, how are you?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "Can you tell me the color of the ocean?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "What year was Celine Dion born in?"

# Print Claude's response
print(get_completion(PROMPT))

现在让我们看看一些不包含正确 Messages API 格式的提示。对于这些格式错误的提示，Messages API 会返回错误。

首先，我们有一个 Messages API 调用的示例，它在 `messages` 数组中缺少 `role` 和 `content` 字段。

In [ ]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"Hi Claude, how are you?"}
        ]
    )

# Print Claude's response
print(response[0].text)

这是一个未能在 `user` 和 `assistant` 角色之间交替的提示。

In [ ]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": "What year was Celine Dion born in?"},
          {"role": "user", "content": "Also, can you tell me some other facts about her?"}
        ]
    )

# Print Claude's response
print(response[0].text)

`user` 和 `assistant` 消息**必须交替**，并且消息**必须以 `user` 轮次开始**。您可以在提示中有多个 `user` 和 `assistant` 对（就像模拟多轮对话一样）。您还可以在终端 `assistant` 消息中放入单词，让 Claude 从您停止的地方继续（在后面的章节中会详细介绍）。

#### 系统提示

您还可以使用**系统提示**。系统提示是一种在"User"轮次中向 Claude 提出问题或任务之前，**为 Claude 提供上下文、指令和指南**的方式。

在结构上，系统提示与 `user` 和 `assistant` 消息列表分开存在，因此属于单独的 `system` 参数（请查看笔记本[设置](#setup)部分中 `get_completion` 辅助函数的结构）。

在本教程中，无论我们可能使用系统提示的地方，我们都在完成函数中为您提供了一个 `system` 字段。如果您不想使用系统提示，只需将 `SYSTEM_PROMPT` 变量设置为空字符串即可。

#### 系统提示示例

In [ ]:
# System prompt
SYSTEM_PROMPT = "Your answer should always be a series of critical thinking questions that further the conversation (do not provide answers to your questions). Do not actually answer the user question."

# Prompt
PROMPT = "Why is the sky blue?"

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))

为什么使用系统提示？**编写良好的系统提示可以在多个方面提高 Claude 的性能**，例如增强 Claude 遵循规则和指令的能力。有关更多信息，请访问我们关于[如何使用系统提示](https://docs.anthropic.com/claude/docs/how-to-use-system-prompts)的文档。

现在我们将深入一些练习。如果您想在不更改上面任何内容的情况下尝试课程提示，请向下滚动到课程笔记本底部访问[**示例演练场**](#example-playground)。

---

## 练习
- [练习 1.1 - 数到三](#exercise-11---counting-to-three)
- [练习 1.2 - 系统提示](#exercise-12---system-prompt)

### 练习 1.1 - 数到三
使用正确的 `user` / `assistant` 格式，编辑下面的 `PROMPT` 以让 Claude **数到三**。输出还将指示您的解决方案是否正确。

In [ ]:
# Prompt - this is the only field you should change
PROMPT = "[Replace this text]"

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    pattern = re.compile(r'^(?=.*1)(?=.*2)(?=.*3).*$', re.DOTALL)
    return bool(pattern.match(text))

# Print Claude's response and the corresponding grade
print(response)
print("\n--------------------------- GRADING ---------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ 如果您需要提示，请运行下面的单元格！

In [ ]:
from hints import exercise_1_1_hint; print(exercise_1_1_hint)

### 练习 1.2 - 系统提示

修改 `SYSTEM_PROMPT` 使 Claude 的回答像一个 3 岁的孩子。

In [ ]:
# System prompt - this is the only field you should change
SYSTEM_PROMPT = "[Replace this text]"

# Prompt
PROMPT = "How big is the sky?"

# Get Claude's response
response = get_completion(PROMPT, SYSTEM_PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search(r"giggles", text) or re.search(r"soo", text))

# Print Claude's response and the corresponding grade
print(response)
print("\n--------------------------- GRADING ---------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ 如果您需要提示，请运行下面的单元格！

In [ ]:
from hints import exercise_1_2_hint; print(exercise_1_2_hint)

### 恭喜！

如果您已经解决了到目前为止的所有练习，那么您已经准备好进入下一章了。祝您提示愉快！

---

## 示例演练场

这是一个供您自由尝试本课程中显示的提示示例并调整提示以查看它如何影响 Claude 响应的区域。

In [ ]:
# Prompt
PROMPT = "Hi Claude, how are you?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "Can you tell me the color of the ocean?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "What year was Celine Dion born in?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"Hi Claude, how are you?"}
        ]
    )

# Print Claude's response
print(response[0].text)

In [ ]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": "What year was Celine Dion born in?"},
          {"role": "user", "content": "Also, can you tell me some other facts about her?"}
        ]
    )

# Print Claude's response
print(response[0].text)

In [ ]:
# System prompt
SYSTEM_PROMPT = "Your answer should always be a series of critical thinking questions that further the conversation (do not provide answers to your questions). Do not actually answer the user question."

# Prompt
PROMPT = "Why is the sky blue?"

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))